In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import joblib
import requests
import io
import warnings
from datetime import datetime, timedelta
from sklearn.preprocessing import LabelEncoder, MinMaxScaler

# Suppress specific pandas downcasting warnings
warnings.filterwarnings("ignore", category=FutureWarning, module="pandas")

# ==========================================
# 0. CONFIGURATION & MAPPING
# ==========================================
ODDS_API_KEY = "d991f2b97d765e32f215b6e510bfae87"
SPORT = "soccer_epl"
REGION = "uk"
MARKET = "h2h"

TEAM_NAME_MAP = {
    "Manchester City FC": "Man City", "Manchester City": "Man City",
    "Manchester United FC": "Man United", "Manchester United": "Man United",
    "Tottenham Hotspur FC": "Tottenham", "Tottenham Hotspur": "Tottenham",
    "Arsenal FC": "Arsenal", "Arsenal": "Arsenal",
    "Liverpool FC": "Liverpool", "Liverpool": "Liverpool",
    "Chelsea FC": "Chelsea", "Chelsea": "Chelsea",
    "Aston Villa FC": "Aston Villa", "Aston Villa": "Aston Villa",
    "Newcastle United FC": "Newcastle", "Newcastle United": "Newcastle",
    "Brighton & Hove Albion FC": "Brighton", "Brighton and Hove Albion": "Brighton",
    "Brentford FC": "Brentford", "Brentford": "Brentford",
    "West Ham United FC": "West Ham", "West Ham United": "West Ham",
    "Crystal Palace FC": "Crystal Palace", "Crystal Palace": "Crystal Palace",
    "Fulham FC": "Fulham", "Fulham": "Fulham",
    "AFC Bournemouth": "Bournemouth", "Bournemouth": "Bournemouth",
    "Everton FC": "Everton", "Everton": "Everton",
    "Nottingham Forest FC": "Nott'm Forest", "Nottingham Forest": "Nott'm Forest",
    "Wolverhampton Wanderers FC": "Wolves", "Wolverhampton Wanderers": "Wolves",
    "Leicester City FC": "Leicester", "Leicester City": "Leicester",
    "Ipswich Town FC": "Ipswich", "Ipswich Town": "Ipswich",
    "Southampton FC": "Southampton", "Southampton": "Southampton"
}

# ==========================================
# 1. DATA FETCHING
# ==========================================

def fetch_real_time_odds():
    url = f"https://api.the-odds-api.com/v4/sports/{SPORT}/odds"
    params = {'apiKey': ODDS_API_KEY, 'regions': REGION, 'markets': MARKET, 'oddsFormat': 'decimal'}
    try:
        response = requests.get(url, params=params)
        if response.status_code != 200: return {}
        data = response.json()
        odds_lookup = {}
        for match in data:
            h_norm = TEAM_NAME_MAP.get(match['home_team'], match['home_team'])
            a_norm = TEAM_NAME_MAP.get(match['away_team'], match['away_team'])
            h_odds, d_odds, a_odds = [], [], []
            for bookie in match['bookmakers']:
                m_data = next((m for m in bookie['markets'] if m['key'] == 'h2h'), None)
                if m_data:
                    for outcome in m_data['outcomes']:
                        if outcome['name'] == match['home_team']: h_odds.append(outcome['price'])
                        elif outcome['name'] == match['away_team']: a_odds.append(outcome['price'])
                        elif outcome['name'] == 'Draw': d_odds.append(outcome['price'])
            if h_odds and d_odds and a_odds:
                odds_lookup[f"{h_norm}|{a_norm}"] = (np.mean(h_odds), np.mean(d_odds), np.mean(a_odds))
        return odds_lookup
    except: return {}

def fetch_upcoming_fixtures(api_key="002a75212e3f409386e933814be7ced1"):
    url = "https://api.football-data.org/v4/competitions/PL/matches"
    headers = {'X-Auth-Token': api_key}
    params = {'status': 'SCHEDULED'}
    live_odds_data = fetch_real_time_odds()
    try:
        response = requests.get(url, headers=headers, params=params)
        data = response.json()
        all_matches = data.get('matches', [])
        if not all_matches: return pd.DataFrame()
        all_matches.sort(key=lambda x: x['utcDate'])
        next_10 = all_matches[:10]
        fix_rows = []
        for m in next_10:
            m_date = pd.to_datetime(m['utcDate']).tz_localize(None)
            h_name = TEAM_NAME_MAP.get(m['homeTeam']['name'], m['homeTeam']['name'])
            a_name = TEAM_NAME_MAP.get(m['awayTeam']['name'], m['awayTeam']['name'])
            odds_key = f"{h_name}|{a_name}"
            o_h, o_d, o_a = live_odds_data.get(odds_key, (2.30, 3.30, 3.00))
            fix_rows.append({
                "Date": m_date, "HomeTeam": h_name, "AwayTeam": a_name,
                "AvgH": round(o_h, 2), "AvgD": round(o_d, 2), "AvgA": round(o_a, 2), "FTR": "U"
            })
        return pd.DataFrame(fix_rows)
    except: return pd.DataFrame()

def fetch_historical_data_football_uk():
    now = datetime.now()
    season = f"{str(now.year-1)[2:]}{str(now.year)[2:]}" if now.month < 8 else f"{str(now.year)[2:]}{str(now.year+1)[2:]}"
    url = f"https://www.football-data.co.uk/mmz4281/{season}/E0.csv"
    cols = ['Date', 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG', 'FTR', 'HTHG', 'HTAG', 'HS', 'AS', 'HST', 'AST', 'HC', 'AC', 'HF', 'AF', 'HY', 'AY', 'HR', 'AR', 'AvgH', 'AvgD', 'AvgA']
    try:
        r = requests.get(url)
        df = pd.read_csv(io.StringIO(r.text))
        df = df[[c for c in cols if c in df.columns]]
        df['Date'] = pd.to_datetime(df['Date'], dayfirst=True, errors='coerce').dt.tz_localize(None)
        return df.dropna(subset=['Date', 'HomeTeam', 'AwayTeam'])
    except: return None

# ==========================================
# 2. FEATURE ENGINEERING
# ==========================================

def calculate_h2h_stats(df):
    df = df.sort_values('Date').reset_index(drop=True)
    df['h2h_id'] = df.apply(lambda x: f"{min(str(x['HomeTeam']), str(x['AwayTeam']))}_{max(str(x['HomeTeam']), str(x['AwayTeam']))}", axis=1)
    df['h2h_home_win_rate'], df['h2h_draw_rate'] = 0.33, 0.33
    for _, group in df.groupby('h2h_id'):
        past = group['FTR'].shift(1)
        count = np.arange(len(group))
        mask = count > 0
        df.loc[group.index[mask], 'h2h_home_win_rate'] = (past == 'H').cumsum()[mask] / count[mask]
        df.loc[group.index[mask], 'h2h_draw_rate'] = (past == 'D').cumsum()[mask] / count[mask]
    return df

def get_stats_vector(row):
    stat_cols = ['FTHG', 'FTAG', 'HTHG', 'HTAG', 'HS', 'AS', 'HST', 'AST', 'HC', 'AC', 'HF', 'AF', 'HY', 'AY', 'HR', 'AR']
    return row[stat_cols].fillna(0).values.astype(np.float32)

def prepare_inference_data(df_history, df_upcoming, window=5):
    le = LabelEncoder()
    all_teams = pd.concat([df_history['HomeTeam'], df_history['AwayTeam'], df_upcoming['HomeTeam'], df_upcoming['AwayTeam']]).unique()
    le.fit(all_teams)
    df_full = pd.concat([df_history, df_upcoming], ignore_index=True)
    df_full['HomeTeam_ID'], df_full['AwayTeam_ID'] = le.transform(df_full['HomeTeam']), le.transform(df_full['AwayTeam'])
    df_full = calculate_h2h_stats(df_full)
    to_predict = df_full[df_full['FTR'] == 'U'].copy()
    seqs, stats, fixs = [], [], []
    
    # We define a null vector for padding if a team doesn't have enough history
    null_vector = np.zeros(16, dtype=np.float32)
    
    for _, match in to_predict.iterrows():
        h_id, a_id = match['HomeTeam'], match['AwayTeam']
        history_pool = df_full[df_full['Date'] < match['Date']]
        
        h_h = history_pool[(history_pool['HomeTeam'] == h_id) | (history_pool['AwayTeam'] == h_id)].tail(window)
        a_h = history_pool[(history_pool['HomeTeam'] == a_id) | (history_pool['AwayTeam'] == a_id)].tail(window)
        
        # Padding logic: If a team has < 5 matches, pad with zeros so the GRU still gets a 5-step sequence
        h_seq = [get_stats_vector(r) for _, r in h_h.iterrows()]
        while len(h_seq) < window: h_seq.insert(0, null_vector)
        
        a_seq = [get_stats_vector(r) for _, r in a_h.iterrows()]
        while len(a_seq) < window: a_seq.insert(0, null_vector)
        
        seqs.append(np.hstack([np.array(h_seq), np.array(a_seq)]))
        stats.append([match['HomeTeam_ID'], match['AwayTeam_ID'], match['AvgH'], match['AvgD'], match['AvgA'], match['h2h_home_win_rate'], match['h2h_draw_rate']])
        fixs.append(match)
        
    return np.array(seqs), np.array(stats), pd.DataFrame(fixs)

# ==========================================
# 3. MODEL & RUN
# ==========================================

class FootballGRU(nn.Module):
    def __init__(self, input_size=32, hidden_size=32, num_classes=3, num_layers=1):
        super().__init__()
        self.hidden_size, self.num_layers = hidden_size, num_layers
        self.gru = nn.GRU(input_size, hidden_size, num_layers=num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, num_classes)
    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        _, hn = self.gru(x, h0); latent = hn[-1]
        return self.fc(latent), latent

def run_pipeline():
    MODEL_DIR = "models"
    GRU_PATH, XGB_PATH = os.path.join(MODEL_DIR, "final_model_GRU_5.pkl"), os.path.join(MODEL_DIR, "best_model_6_tuned_xgboost.pkl")
    df_hist, df_upcoming = fetch_historical_data_football_uk(), fetch_upcoming_fixtures()
    if df_hist is None or df_upcoming.empty: return
    match_seqs, static_facts, fixtures_df = prepare_inference_data(df_hist, df_upcoming)
    if len(match_seqs) == 0: return

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    gru_model = FootballGRU().to(device)
    loaded = joblib.load(GRU_PATH)
    gru_model.load_state_dict(loaded if isinstance(loaded, dict) else loaded.state_dict()); gru_model.eval()

    with torch.no_grad():
        _, momentum_raw = gru_model(torch.tensor(match_seqs, dtype=torch.float32).to(device))
        momentum = momentum_raw.cpu().numpy()

    # --- ADVANCED UI CATEGORIES ---
    scaler = MinMaxScaler(feature_range=(0, 100))
    m_scaled = scaler.fit_transform(momentum)
    
    fixtures_df['H_Attacking'] = np.mean(m_scaled[:, 0:4], axis=1).round(1)
    fixtures_df['H_Defending'] = np.mean(m_scaled[:, 4:8], axis=1).round(1)
    fixtures_df['H_Volatility'] = np.mean(m_scaled[:, 8:12], axis=1).round(1)
    fixtures_df['H_Efficiency'] = np.mean(m_scaled[:, 12:16], axis=1).round(1)
    
    fixtures_df['A_Attacking'] = np.mean(m_scaled[:, 16:20], axis=1).round(1)
    fixtures_df['A_Defending'] = np.mean(m_scaled[:, 20:24], axis=1).round(1)
    fixtures_df['A_Volatility'] = np.mean(m_scaled[:, 24:28], axis=1).round(1)
    fixtures_df['A_Efficiency'] = np.mean(m_scaled[:, 28:32], axis=1).round(1)

    xgb_model = joblib.load(XGB_PATH)
    probs = xgb_model.predict_proba(np.hstack([static_facts, momentum]))
    
    # Probabilities
    fixtures_df['Prob_Away'] = probs[:, 0].round(3)
    fixtures_df['Prob_Draw'] = probs[:, 1].round(3)
    fixtures_df['Prob_Home'] = probs[:, 2].round(3)
    
    outcome_map = {0: 'Away Win', 1: 'Draw', 2: 'Home Win'}
    fixtures_df['Outcome'] = [outcome_map[i] for i in np.argmax(probs, axis=1)]
    fixtures_df['Conf'] = np.max(probs, axis=1).round(3)

    # --- CSV CLEANUP: ONLY KEEP NECESSARY COLUMNS ---
    clean_cols = [
        'Date', 'HomeTeam', 'AwayTeam', 'HomeTeam_ID', 'AwayTeam_ID',
        'AvgH', 'AvgD', 'AvgA',
        'H_Attacking', 'H_Defending', 'H_Volatility', 'H_Efficiency',
        'A_Attacking', 'A_Defending', 'A_Volatility', 'A_Efficiency',
        'Prob_Home', 'Prob_Draw', 'Prob_Away', 'Outcome', 'Conf'
    ]
    
    final_output = fixtures_df[clean_cols].copy()
    final_output.to_csv("epl_predictions.csv", index=False)
    print(f"Cleaned predictions saved with {len(final_output.columns)} columns.")

    # Terminal Summary
    print("\n" + "="*95)
    print(f"{'DATE':<14} | {'MATCH':<28} | {'VOLATILITY':<12} | {'EFFICIENCY':<12} | {'PICK'}")
    print("-" * 95)
    for _, row in final_output.sort_values('Date').iterrows():
        match_str = f"{row['HomeTeam']} v {row['AwayTeam']}"
        vol_str = f"H:{row['H_Volatility']:.0f} A:{row['A_Volatility']:.0f}"
        eff_str = f"H:{row['H_Efficiency']:.0f} A:{row['A_Efficiency']:.0f}"
        print(f"{row['Date'].strftime('%d %b %H:%M'):<14} | {match_str:<28} | {vol_str:<12} | {eff_str:<12} | {row['Outcome']} ({row['Conf']:.1%})")

if __name__ == "__main__":
    run_pipeline()